# Biohub: train once, compare four checkpoints

This is a **standalone Kaggle notebook for a fresh session**. Attach the Biohub competition data and enable **GPU** and **Internet**, then run all cells. It clones the pushed `biohub-io` branch, hides about 20% of GT nodes (seed 42), trains on `visible.geff` for **2,000 iterations**, and saves checkpoints at 500, 1000, 1500, and 2000. It reconstructs the same 20-frame ROI at iteration 2000, calibrates ILP from visible GT only, then runs inference for checkpoints 500, 1000, and 1500 with the same ROI and fixed ILP costs. The 2000 prediction is reused in the comparison. The last cell displays a table across iterations. All outputs are saved under `/kaggle/working/biohub_random_hide/`.

This rerun uses new random training batches, so its metrics may differ from the previous Kaggle session even with the same seed. Hidden-GT metrics are exploratory diagnostics; choose future training length from the visible candidate trend, not the maximum hidden score.

In [ ]:
from pathlib import Path
import json, subprocess, sys
DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
ROOT = Path('/kaggle/working/biohub_random_hide')
REPO = Path('/kaggle/working/linajea')
SAMPLE = '6bba_05b6850b'
HIDE_RATE = 0.20
SEED = 42
ITERATIONS = 2000
BRANCH = 'biohub-io'
PINNED_COMMIT = None  # Set to the SHA after pushing for a fixed rerun
if not (DATA_DIR / f'{SAMPLE}.zarr').is_dir() or not (DATA_DIR / f'{SAMPLE}.geff').is_dir():
    raise FileNotFoundError(f'Missing Zarr/GEFF pair for {SAMPLE} in {DATA_DIR}')
ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', DATA_DIR, 'Sample:', SAMPLE, 'Output:', ROOT)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU for this training run')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hathanhv/linajea.git', str(REPO)], check=True)
COMMIT = PINNED_COMMIT or subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', COMMIT], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(ROOT / 'commit.txt').write_text(COMMIT + '\n')
print('Pinned code commit:', COMMIT, 'GPU:', torch.cuda.get_device_name(0))
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *map(str, args)], check=True)
pip('-r', REPO / 'requirements-biohub-io.txt')
pip('-r', REPO / 'requirements-biohub-train.txt')
pip('--no-deps', 'git+https://github.com/abred/daisy@ab7b82054126a50ab4ad5dcf2ef885b02d33aab8')
pip('--no-deps', 'git+https://github.com/Kainmueller-Lab/funlib.learn.torch@36ef666d16bddda0bf3146d4c0efbe9b95c8463d')
pip('-e', REPO, '--no-deps')
if not (REPO / 'run_scripts' / 'compare_biohub_checkpoints.py').is_file():
    raise FileNotFoundError('Pushed branch is missing checkpoint comparison code')

In [ ]:
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide', 'split',
                '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                '--output-dir', str(ROOT), '--hide-rate', str(HIDE_RATE),
                '--seed', str(SEED)], cwd=REPO, check=True)
split = json.loads((ROOT / 'split.json').read_text())
print('Visible nodes:', len(split['visible_node_ids']),
      'Hidden nodes:', len(split['hidden_node_ids']),
      'Visible edges:', len(split['visible_edges']),
      'Hidden edges:', len(split['hidden_edges']))
print('Pilot ROI start / shape:', split['roi_start'], split['roi_shape'])

In [ ]:
TRAIN_DIR = ROOT / 'train'
TRAIN_DIR.mkdir(exist_ok=True)
command = [sys.executable, '-m', 'run_scripts.train_biohub_smoke',
           '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
           '--tracksfile', str(ROOT / 'visible.geff'),
           '--output-dir', str(TRAIN_DIR),
           '--iterations', str(ITERATIONS), '--checkpoint-stride', '500']
with (TRAIN_DIR / 'train.log').open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line); log.flush()
        if 'Batch: iteration=' in line:
            iteration = int(line.split('Batch: iteration=')[1].split(',')[0])
            if iteration == 1 or iteration % 100 == 0:
                print(line.rstrip())
        elif 'checkpoint' in line.lower() or 'Traceback' in line:
            print(line.rstrip())
    if proc.wait() != 0:
        raise RuntimeError(f'Training failed; inspect {TRAIN_DIR / "train.log"}')
checkpoint = TRAIN_DIR / f'train_net_checkpoint_{ITERATIONS}'
print('Checkpoint saved:', checkpoint, checkpoint.stat().st_size, 'bytes')
for iteration in (500, 1000, 1500, 2000):
    path = TRAIN_DIR / f'train_net_checkpoint_{iteration}'
    if not path.is_file():
        raise FileNotFoundError(f'Missing saved checkpoint: {path}')
print('All four comparison checkpoints found.')

In [ ]:
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide',
                'reconstruct', '--data-dir', str(DATA_DIR),
                '--sample', SAMPLE, '--output-dir', str(ROOT),
                '--iterations', str(ITERATIONS)], cwd=REPO, check=True)
print('Prediction files:', sorted(p.name for p in (ROOT / 'prediction').iterdir()))

In [ ]:
sample = SAMPLE
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide', 'calibrate',
                '--data-dir', str(DATA_DIR), '--sample', sample,
                '--output-dir', str(ROOT)], cwd=REPO, check=True)
report = json.loads((ROOT / 'prediction_calibrated' / 'calibration.json').read_text())
print('Candidate-supported visible nodes/edges:',
      report['candidate_supported_visible_nodes'],
      report['candidate_supported_visible_edges'])
print('Chosen ILP costs and visible recovery:', report['chosen'])
print('Full grid:', ROOT / 'prediction_calibrated' / 'selection_grid.csv')

In [ ]:
from IPython.display import display
import pandas as pd
comparison_dir = ROOT / 'checkpoint_comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)
(comparison_dir / 'code_commit.txt').write_text(COMMIT + '\n')
ITERATIONS = (500, 1000, 1500, 2000)
command = [sys.executable, '-m', 'run_scripts.compare_biohub_checkpoints',
           '--data-dir', str(DATA_DIR), '--output-dir', str(ROOT),
           '--iterations', *map(str, ITERATIONS),
           '--reuse-iteration', '2000', '--device', 'cuda']
with (comparison_dir / 'run.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log.write(line)
        log.flush()
        if ('Iteration ' in line or 'Traceback' in line or 'Error' in line):
            print(line.rstrip())
    if process.wait() != 0:
        raise RuntimeError(f'Checkpoint comparison failed. See {comparison_dir / "run.log"}')
print('Comparison saved to:', comparison_dir)

In [ ]:
# Final comparison table at the fixed 4 µm match distance.
summary = pd.read_csv(comparison_dir / 'summary_4um.csv')
all_metrics = pd.read_csv(comparison_dir / 'metrics_all_thresholds.csv')
visible_total = int(all_metrics['visible_total'].iloc[0])
hidden_total = int(all_metrics['hidden_total'].iloc[0])
hidden_edge_total = int(all_metrics['hidden_edges_total'].iloc[0])

def fraction(value, total):
    return f'{int(value)}/{total} ({value / total:.0%})' if total else 'n/a'

table = pd.DataFrame({
    'Iteration': summary['iteration'],
    'Candidate nodes': summary['candidate_nodes'],
    'Visible GT matched (candidate)': [
        fraction(v, visible_total) for v in summary['candidate_visible_recovered_4um']],
    'Hidden GT matched (candidate)': [
        fraction(v, hidden_total) for v in summary['candidate_hidden_recovered_4um']],
    'Selected nodes': summary['selected_nodes'],
    'Hidden GT matched (after ILP)': [
        fraction(v, hidden_total) for v in summary['selected_hidden_recovered_4um']],
    'Hidden edges (candidate)': [
        fraction(v, hidden_edge_total) for v in summary['candidate_hidden_edges_4um']],
    'Hidden edges (after ILP)': [
        fraction(v, hidden_edge_total) for v in summary['selected_hidden_edges_4um']],
    'Mean candidate score': summary['candidate_score_mean'].round(3),
})
display(table.set_index('Iteration'))
report = json.loads((comparison_dir / 'comparison.json').read_text())
print('Visible-only diagnostic checkpoint:',
      report['suggested_iteration_from_visible_only'])
print('Full 2/4/7 µm metrics:', comparison_dir / 'metrics_all_thresholds.csv')
print('This is a same-sequence exploratory comparison, not independent validation.')